# Prototipo de la métrica Macro AP-rIoU

Este notebook es el espacio de experimentación de la issue #9. Saúl y Dolly lo usarán para comprender y comprobar manualmente cada parte de la métrica antes de trasladarla a `src/evaluation/metric.py`.

**Estado actual:** el pipeline matemático completo fue validado localmente y en Google Colab. Su implementación modular y tipada ya fue migrada a `src/evaluation/metric.py`; este notebook conserva la explicación experimental reproducible.

## 1. Contrato que debemos respetar

Una OBB se representa como `(cx, cy, width, height, angle_deg)`. El centro `(cx, cy)`, el ancho y el alto se expresan en píxeles; el ángulo se expresa en grados.

Una predicción completa usa `(frame_id, score, cx, cy, width, height, angle_deg)`. El ground truth no contiene `score` porque es la respuesta correcta y no una estimación del modelo.

En este primer experimento la entrada es una OBB paramétrica y la salida esperada son sus cuatro vértices en píxeles.

In [ ]:
import math

import cv2
import numpy as np
from IPython.display import Image, display

## 2. De parámetros a cuatro vértices

Primero colocamos cuatro esquinas alrededor del origen: `(-w/2, -h/2)`, `(w/2, -h/2)`, `(w/2, h/2)` y `(-w/2, h/2)`. Después las rotamos por `angle_deg` y finalmente trasladamos todas al centro `(cx, cy)`.

La función siguiente es deliberadamente experimental. Cuando entendamos y validemos todos sus casos, la implementación definitiva se escribirá y probará en `metric.py`.

In [ ]:
def experimental_obb_to_vertices(obb):
    cx, cy, width, height, angle_deg = obb
    theta = math.radians(angle_deg)
    rotation = np.array(
        [
            [math.cos(theta), -math.sin(theta)],
            [math.sin(theta), math.cos(theta)],
        ],
        dtype=np.float64,
    )
    local_vertices = np.array(
        [
            [-width / 2, -height / 2],
            [width / 2, -height / 2],
            [width / 2, height / 2],
            [-width / 2, height / 2],
        ],
        dtype=np.float64,
    )
    return local_vertices @ rotation.T + np.array([cx, cy])


axis_aligned_obb = (200.0, 150.0, 120.0, 60.0, 0.0)
axis_aligned_vertices = experimental_obb_to_vertices(axis_aligned_obb)
expected_vertices = np.array(
    [[140.0, 120.0], [260.0, 120.0], [260.0, 180.0], [140.0, 180.0]]
)

assert np.allclose(axis_aligned_vertices, expected_vertices)
assert math.isclose(cv2.contourArea(axis_aligned_vertices.astype(np.float32)), 120.0 * 60.0)

print("Vértices calculados:")
print(axis_aligned_vertices)
print()
print("Comprobación: área del polígono = width × height = 7200 px²")

## 3. Visualización de una OBB rotada

Ahora conservamos el mismo centro, ancho y alto, pero usamos un ángulo de 30°. La rotación debe cambiar los vértices sin cambiar el centro ni el área.

In [ ]:
rotated_obb = (200.0, 150.0, 120.0, 60.0, 30.0)
rotated_vertices = experimental_obb_to_vertices(rotated_obb)
rotated_area = cv2.contourArea(rotated_vertices.astype(np.float32))

assert math.isclose(rotated_area, 120.0 * 60.0, rel_tol=1e-6)
assert np.allclose(rotated_vertices.mean(axis=0), [200.0, 150.0])

canvas = np.full((300, 400, 3), 245, dtype=np.uint8)
polygon = np.rint(rotated_vertices).astype(np.int32).reshape((-1, 1, 2))
cv2.polylines(canvas, [polygon], isClosed=True, color=(40, 120, 220), thickness=3)
cv2.circle(canvas, (200, 150), radius=5, color=(220, 60, 40), thickness=-1)
cv2.putText(
    canvas,
    "centro (200, 150)",
    (210, 145),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.5,
    (40, 40, 40),
    1,
    cv2.LINE_AA,
)
success, encoded_image = cv2.imencode(".png", canvas)
assert success

print(f"Centro recuperado: {rotated_vertices.mean(axis=0)}")
print(f"Área después de rotar: {rotated_area:.1f} px²")
display(Image(data=encoded_image.tobytes()))

## 4. Qué debemos entender antes de continuar

1. `cx` y `cy` indican el centro, no una esquina.
2. Antes de rotar, las esquinas se construyen usando la mitad del ancho y del alto.
3. La rotación cambia la posición de los vértices, pero conserva el centro y el área.
4. Los vértices son necesarios porque rIoU compara la intersección de dos polígonos rotados.

**Siguiente experimento:** validar dimensiones y ángulos problemáticos antes de calcular la intersección entre dos OBB.

## 5. Validación experimental de una OBB

No todas las tuplas de cinco números describen una caja geométrica válida. El ancho y el alto deben ser estrictamente positivos; además, el centro, las dimensiones y el ángulo deben ser valores finitos. NaN e infinito harían imposible obtener vértices confiables.

In [ ]:
def experimental_validate_obb(obb):
    values = np.asarray(obb, dtype=np.float64)
    if values.shape != (5,):
        raise ValueError("Una OBB debe contener exactamente cinco valores")
    if not np.all(np.isfinite(values)):
        raise ValueError("Todos los valores de la OBB deben ser finitos")
    if values[2] <= 0 or values[3] <= 0:
        raise ValueError("El ancho y el alto deben ser mayores que cero")
    return tuple(float(value) for value in values)


valid_obb = (200.0, 150.0, 120.0, 60.0, 30.0)
assert experimental_validate_obb(valid_obb) == valid_obb

invalid_obbs = {
    "ancho cero": (200.0, 150.0, 0.0, 60.0, 30.0),
    "altura negativa": (200.0, 150.0, 120.0, -60.0, 30.0),
    "centro NaN": (math.nan, 150.0, 120.0, 60.0, 30.0),
    "ángulo infinito": (200.0, 150.0, 120.0, 60.0, math.inf),
}

for case_name, invalid_obb in invalid_obbs.items():
    try:
        experimental_validate_obb(invalid_obb)
    except ValueError as error:
        print(f"✓ {case_name}: rechazada ({error})")
    else:
        raise AssertionError(f"La OBB inválida '{case_name}' fue aceptada")

## 6. Normalización y equivalencia de ángulos

Una vuelta completa tiene 360°. Por eso, sumar o restar 360° no cambia la orientación de la caja. Normalizar con módulo 360 lleva cualquier ángulo finito al intervalo [0°, 360°). Por ejemplo, -15° se convierte en 345°.

In [ ]:
def experimental_normalize_angle(angle_deg):
    if not math.isfinite(angle_deg):
        raise ValueError("El ángulo debe ser finito")
    return angle_deg % 360.0


negative_angle = -15.0
normalized_angle = experimental_normalize_angle(negative_angle)
assert normalized_angle == 345.0

negative_angle_obb = (200.0, 150.0, 120.0, 60.0, negative_angle)
normalized_angle_obb = (200.0, 150.0, 120.0, 60.0, normalized_angle)
negative_vertices = experimental_obb_to_vertices(negative_angle_obb)
normalized_vertices = experimental_obb_to_vertices(normalized_angle_obb)

assert np.allclose(negative_vertices, normalized_vertices, atol=1e-9)

print(f"{negative_angle}° normalizado = {normalized_angle}°")
print(
    "Diferencia máxima entre sus vértices:",
    float(np.max(np.abs(negative_vertices - normalized_vertices))),
)

## 7. Conclusiones de este experimento

1. Una dimensión igual a cero no forma una superficie y debe rechazarse.
2. Una dimensión negativa no tiene interpretación geométrica y debe rechazarse.
3. NaN e infinito contaminarían todos los cálculos posteriores.
4. -15° y 345° representan la misma orientación y producen los mismos vértices.
5. Estas reglas se trasladarán después a validate_obb() y normalize_angle() dentro de metric.py.

**Siguiente experimento:** dibujar dos OBB superpuestas y comprender visualmente intersección, unión y rIoU.

## 8. Intersección, unión y rIoU

La intersección es el área cubierta simultáneamente por las dos OBB. La unión es toda el área cubierta por al menos una de ellas y se calcula como área A + área B - intersección. Finalmente, rIoU = intersección / unión.

Usaremos primero un ejemplo comprobable a mano. Cada caja mide 120 × 80, por lo que su área es 9600 px². Al desplazar la segunda caja 40 px horizontalmente, comparten una región de 80 × 80 = 6400 px². La unión es 9600 + 9600 - 6400 = 12800 px² y rIoU debe ser 6400 / 12800 = 0.5.

In [ ]:
def experimental_rotated_iou(obb_a, obb_b):
    valid_a = experimental_validate_obb(obb_a)
    valid_b = experimental_validate_obb(obb_b)
    polygon_a = experimental_obb_to_vertices(valid_a).astype(np.float32)
    polygon_b = experimental_obb_to_vertices(valid_b).astype(np.float32)

    area_a = float(cv2.contourArea(polygon_a))
    area_b = float(cv2.contourArea(polygon_b))
    intersection_area, intersection_polygon = cv2.intersectConvexConvex(
        polygon_a, polygon_b
    )
    intersection_area = float(intersection_area)
    union_area = area_a + area_b - intersection_area
    if union_area <= 0.0:
        return 0.0, intersection_area, union_area, intersection_polygon

    riou = min(1.0, max(0.0, intersection_area / union_area))
    return riou, intersection_area, union_area, intersection_polygon


box_a = (160.0, 150.0, 120.0, 80.0, 0.0)
box_b = (200.0, 150.0, 120.0, 80.0, 0.0)
riou, intersection_area, union_area, intersection_polygon = (
    experimental_rotated_iou(box_a, box_b)
)

assert math.isclose(intersection_area, 6400.0)
assert math.isclose(union_area, 12800.0)
assert math.isclose(riou, 0.5)

print(f"Intersección: {intersection_area:.1f} px²")
print(f"Unión: {union_area:.1f} px²")
print(f"rIoU: {riou:.2f}")

In [ ]:
canvas = np.full((300, 400, 3), 245, dtype=np.uint8)
vertices_a = experimental_obb_to_vertices(box_a)
vertices_b = experimental_obb_to_vertices(box_b)
polygon_a_draw = np.rint(vertices_a).astype(np.int32).reshape((-1, 1, 2))
polygon_b_draw = np.rint(vertices_b).astype(np.int32).reshape((-1, 1, 2))

if intersection_polygon is not None and len(intersection_polygon) >= 3:
    intersection_draw = np.rint(intersection_polygon).astype(np.int32)
    overlay = canvas.copy()
    cv2.fillPoly(overlay, [intersection_draw], color=(90, 200, 90))
    canvas = cv2.addWeighted(overlay, 0.55, canvas, 0.45, 0.0)

cv2.polylines(canvas, [polygon_a_draw], True, (220, 80, 60), 3)
cv2.polylines(canvas, [polygon_b_draw], True, (60, 100, 220), 3)
cv2.circle(canvas, (160, 150), 4, (220, 80, 60), -1)
cv2.circle(canvas, (200, 150), 4, (60, 100, 220), -1)
cv2.putText(canvas, "A", (145, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (220, 80, 60), 2)
cv2.putText(canvas, "B", (215, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (60, 100, 220), 2)

success, encoded_intersection = cv2.imencode(".png", canvas)
assert success
display(Image(data=encoded_intersection.tobytes()))

## 9. Casos extremos de rIoU

Una implementación correcta debe devolver 1 para cajas idénticas, 0 para cajas separadas y el mismo resultado sin importar el orden de las cajas. En una caja no cuadrada, cambiar únicamente su ángulo debe reducir el solapamiento.

In [ ]:
identical_riou = experimental_rotated_iou(box_a, box_a)[0]
separated_box = (340.0, 150.0, 120.0, 80.0, 0.0)
separated_riou = experimental_rotated_iou(box_a, separated_box)[0]
reverse_riou = experimental_rotated_iou(box_b, box_a)[0]
rotated_box = (160.0, 150.0, 120.0, 80.0, 30.0)
angular_deviation_riou = experimental_rotated_iou(box_a, rotated_box)[0]
equivalent_angle_riou = experimental_rotated_iou(
    negative_angle_obb, normalized_angle_obb
)[0]

assert math.isclose(identical_riou, 1.0)
assert math.isclose(separated_riou, 0.0)
assert math.isclose(reverse_riou, riou)
assert 0.0 < angular_deviation_riou < 1.0
assert math.isclose(equivalent_angle_riou, 1.0, rel_tol=1e-6)

print(f"Cajas idénticas: {identical_riou:.2f}")
print(f"Cajas separadas: {separated_riou:.2f}")
print(f"Orden inverso: {reverse_riou:.2f}")
print(f"Desviación angular de 30°: {angular_deviation_riou:.4f}")
print(f"Ángulos -15° y 345°: {equivalent_angle_riou:.2f}")

## 10. Conclusiones del experimento de rIoU

1. rIoU mide cuánto se superponen dos OBB en relación con toda el área que cubren.
2. Restar la intersección al sumar ambas áreas evita contar dos veces la zona compartida.
3. El resultado siempre debe permanecer entre 0 y 1.
4. Rotar una caja no cuadrada puede reducir rIoU aunque conserve el mismo centro y área.
5. La implementación definitiva se migrará posteriormente a rotated_iou() dentro de metric.py.

**Siguiente experimento:** simular el matching uno a uno entre predicciones ordenadas por score y ground truths.

## 11. Matching greedy uno a uno

Las predicciones se procesan de mayor a menor score. Cada una solo puede compararse con GT del mismo frame y clase. Entre los GT todavía libres se elige el de mayor rIoU; si alcanza el umbral, la predicción es TP y ese GT queda ocupado. Si no existe una coincidencia disponible, la predicción es FP. Al terminar, cada GT no utilizado cuenta como FN.

Greedy significa que cada decisión se toma y se conserva inmediatamente. Una predicción posterior no puede quitarle un GT a una predicción de mayor score.

In [ ]:
def experimental_match_predictions(predictions, ground_truths, riou_threshold):
    if not 0.0 <= riou_threshold <= 1.0:
        raise ValueError("El umbral de rIoU debe estar entre 0 y 1")

    ranked_predictions = sorted(
        enumerate(predictions),
        key=lambda indexed_prediction: -indexed_prediction[1]["score"],
    )
    used_gt_indices = set()
    rows = []
    tp_flags = []
    fp_flags = []

    for original_index, prediction in ranked_predictions:
        best_gt_index = None
        best_riou = 0.0

        for gt_index, ground_truth in enumerate(ground_truths):
            if gt_index in used_gt_indices:
                continue
            if prediction["frame_id"] != ground_truth["frame_id"]:
                continue
            if prediction["class_id"] != ground_truth["class_id"]:
                continue

            candidate_riou = experimental_rotated_iou(
                prediction["obb"], ground_truth["obb"]
            )[0]
            if candidate_riou > best_riou:
                best_riou = candidate_riou
                best_gt_index = gt_index

        is_true_positive = (
            best_gt_index is not None and best_riou >= riou_threshold
        )
        if is_true_positive:
            used_gt_indices.add(best_gt_index)
            selected_gt_id = ground_truths[best_gt_index]["id"]
            tp_flags.append(1)
            fp_flags.append(0)
            outcome = "TP"
        else:
            selected_gt_id = None
            tp_flags.append(0)
            fp_flags.append(1)
            outcome = "FP"

        rows.append(
            {
                "prediction_id": prediction["id"],
                "original_index": original_index,
                "score": prediction["score"],
                "selected_gt_id": selected_gt_id,
                "best_available_riou": best_riou,
                "outcome": outcome,
            }
        )

    tp = sum(tp_flags)
    fp = sum(fp_flags)
    fn = len(ground_truths) - tp
    return {
        "rows": rows,
        "tp_flags": tp_flags,
        "fp_flags": fp_flags,
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }


## 12. Un GT y una predicción duplicada

La lista se entrega deliberadamente desordenada. La predicción de score 0.95 tiene rIoU 0.50 y se procesa primero; como alcanza el umbral 0.50, obtiene el único GT. La predicción de score 0.85 coincide perfectamente con ese GT, pero llega después y debe ser FP porque el GT ya no está libre.

In [ ]:
ground_truths_example = [
    {"id": "gt_1", "frame_id": "frame_001", "class_id": 1, "obb": box_a}
]
predictions_with_duplicate = [
    {
        "id": "pred_duplicate",
        "frame_id": "frame_001",
        "class_id": 1,
        "score": 0.85,
        "obb": box_a,
    },
    {
        "id": "pred_high_score",
        "frame_id": "frame_001",
        "class_id": 1,
        "score": 0.95,
        "obb": box_b,
    },
]

duplicate_result = experimental_match_predictions(
    predictions_with_duplicate, ground_truths_example, riou_threshold=0.50
)

assert [row["prediction_id"] for row in duplicate_result["rows"]] == [
    "pred_high_score",
    "pred_duplicate",
]
assert duplicate_result["tp_flags"] == [1, 0]
assert duplicate_result["fp_flags"] == [0, 1]
assert (duplicate_result["tp"], duplicate_result["fp"], duplicate_result["fn"]) == (
    1,
    1,
    0,
)

for row in duplicate_result["rows"]:
    selected_gt = row["selected_gt_id"] or "ninguno libre"
    print(
        f"{row['prediction_id']}: score={row['score']:.2f}, "
        f"GT={selected_gt}, rIoU disponible={row['best_available_riou']:.2f} "
        f"→ {row['outcome']}"
    )
print(
    f"Conteos: TP={duplicate_result['tp']}, "
    f"FP={duplicate_result['fp']}, FN={duplicate_result['fn']}"
)

## 13. Restricciones por frame y clase, FN y orden estable

Una predicción de otra clase o de otro frame no puede utilizar el GT, aunque tenga la misma geometría. Si ninguna predicción válida utiliza un GT, este queda como FN. Cuando dos predicciones tienen el mismo score, se conserva su orden original para que el resultado sea determinista.

In [ ]:
wrong_context_predictions = [
    {
        "id": "wrong_class",
        "frame_id": "frame_001",
        "class_id": 2,
        "score": 0.90,
        "obb": box_a,
    },
    {
        "id": "wrong_frame",
        "frame_id": "frame_002",
        "class_id": 1,
        "score": 0.80,
        "obb": box_a,
    },
]
wrong_context_result = experimental_match_predictions(
    wrong_context_predictions, ground_truths_example, riou_threshold=0.50
)
empty_result = experimental_match_predictions(
    [], ground_truths_example, riou_threshold=0.50
)

assert (wrong_context_result["tp"], wrong_context_result["fp"], wrong_context_result["fn"]) == (
    0,
    2,
    1,
)
assert (empty_result["tp"], empty_result["fp"], empty_result["fn"]) == (0, 0, 1)

tied_predictions = [
    {**wrong_context_predictions[0], "id": "tie_first", "score": 0.70},
    {**wrong_context_predictions[1], "id": "tie_second", "score": 0.70},
]
tied_result = experimental_match_predictions(
    tied_predictions, ground_truths_example, riou_threshold=0.50
)
assert [row["prediction_id"] for row in tied_result["rows"]] == [
    "tie_first",
    "tie_second",
]

assert duplicate_result["tp"] + duplicate_result["fn"] == len(ground_truths_example)
assert duplicate_result["tp"] + duplicate_result["fp"] == len(
    predictions_with_duplicate
)

print(
    "Frame/clase incorrectos: "
    f"TP={wrong_context_result['tp']}, FP={wrong_context_result['fp']}, "
    f"FN={wrong_context_result['fn']}"
)
print(
    "Sin predicciones: "
    f"TP={empty_result['tp']}, FP={empty_result['fp']}, FN={empty_result['fn']}"
)
print("Empate de score conserva el orden: tie_first → tie_second")

## 14. Conclusiones del experimento de matching

1. El score determina qué predicción intenta emparejarse primero.
2. Cada GT puede utilizarse una sola vez.
3. Un duplicado posterior cuenta como FP aunque su caja sea perfecta.
4. Las comparaciones se restringen al mismo frame y clase.
5. Los GT no utilizados cuentan como FN.
6. Se cumplen las invariantes TP + FN = total GT y TP + FP = total de predicciones.

**Siguiente experimento:** convertir los vectores TP y FP acumulados en Precision y Recall.

## 15. Precision y Recall acumulados

Después del matching, los flags TP y FP ya están ordenados por score. Para cada posición acumulamos cuántos TP y FP hemos visto. Precision indica qué fracción de las predicciones procesadas es correcta; Recall indica qué fracción de todos los GT ya fue recuperada.

Precision(i) = TP acumulados / (TP acumulados + FP acumulados). Recall(i) = TP acumulados / total de GT.

In [ ]:
def experimental_precision_recall(tp_flags, fp_flags, total_gt):
    if len(tp_flags) != len(fp_flags):
        raise ValueError("TP y FP deben tener la misma longitud")
    if total_gt < 0:
        raise ValueError("El total de GT no puede ser negativo")

    tp_array = np.asarray(tp_flags, dtype=np.float64)
    fp_array = np.asarray(fp_flags, dtype=np.float64)
    if not np.all(np.isin(tp_array, [0.0, 1.0])):
        raise ValueError("Los flags TP deben ser binarios")
    if not np.all(np.isin(fp_array, [0.0, 1.0])):
        raise ValueError("Los flags FP deben ser binarios")
    if not np.all(tp_array + fp_array == 1.0):
        raise ValueError("Cada predicción debe ser exactamente TP o FP")

    cumulative_tp = np.cumsum(tp_array)
    cumulative_fp = np.cumsum(fp_array)
    denominators = cumulative_tp + cumulative_fp
    precision = np.divide(
        cumulative_tp,
        denominators,
        out=np.zeros_like(cumulative_tp),
        where=denominators > 0,
    )
    if total_gt == 0:
        recall = np.zeros_like(cumulative_tp)
    else:
        recall = cumulative_tp / float(total_gt)
    return precision, recall, cumulative_tp, cumulative_fp


tp_example = [1, 0, 1, 0]
fp_example = [0, 1, 0, 1]
precision_example, recall_example, cumulative_tp, cumulative_fp = (
    experimental_precision_recall(tp_example, fp_example, total_gt=2)
)

assert np.allclose(cumulative_tp, [1, 1, 2, 2])
assert np.allclose(cumulative_fp, [0, 1, 1, 2])
assert np.allclose(precision_example, [1.0, 0.5, 2.0 / 3.0, 0.5])
assert np.allclose(recall_example, [0.5, 0.5, 1.0, 1.0])

print("posición | TP acum | FP acum | Precision | Recall")
for index in range(len(tp_example)):
    print(
        f"{index + 1:8d} | {cumulative_tp[index]:7.0f} | "
        f"{cumulative_fp[index]:7.0f} | {precision_example[index]:9.4f} | "
        f"{recall_example[index]:6.4f}"
    )

## 16. AP mediante interpolación COCO de 101 puntos

Se evalúan 101 niveles de recall: 0.00, 0.01, ..., 1.00. Para cada nivel se busca la mayor Precision observada en ese nivel de recall o en uno posterior. Si no existe un punto elegible se usa cero. AP es el promedio de esas 101 precisiones interpoladas.

No se usa integración trapezoidal. Si una clase no tiene GT, su AP se define como 0.0.

In [ ]:
def experimental_ap_101(tp_flags, fp_flags, total_gt):
    recall_levels = np.linspace(0.0, 1.0, 101)
    if total_gt == 0:
        zeros = np.zeros_like(recall_levels)
        return 0.0, zeros, zeros, recall_levels, zeros

    precision, recall, _, _ = experimental_precision_recall(
        tp_flags, fp_flags, total_gt
    )
    interpolated_precision = np.zeros_like(recall_levels)
    for index, recall_level in enumerate(recall_levels):
        eligible = precision[recall >= recall_level]
        if eligible.size > 0:
            interpolated_precision[index] = np.max(eligible)

    ap = float(np.mean(interpolated_precision))
    return ap, precision, recall, recall_levels, interpolated_precision


perfect_ap = experimental_ap_101([1], [0], total_gt=1)[0]
empty_ap = experimental_ap_101([], [], total_gt=1)[0]
no_gt_ap = experimental_ap_101([], [], total_gt=0)[0]
tp_then_duplicate_ap = experimental_ap_101([1, 0], [0, 1], total_gt=1)[0]
fp_then_tp_ap = experimental_ap_101([0, 1], [1, 0], total_gt=1)[0]

assert math.isclose(perfect_ap, 1.0)
assert math.isclose(empty_ap, 0.0)
assert math.isclose(no_gt_ap, 0.0)
assert math.isclose(tp_then_duplicate_ap, 1.0)
assert math.isclose(fp_then_tp_ap, 0.5)

print(f"Predicción perfecta: AP={perfect_ap:.2f}")
print(f"Sin predicciones: AP={empty_ap:.2f}")
print(f"Clase sin GT: AP={no_gt_ap:.2f}")
print(f"TP seguido de duplicado FP: AP={tp_then_duplicate_ap:.2f}")
print(f"FP de mayor score antes del TP: AP={fp_then_tp_ap:.2f}")

## 17. AP por clase, siete umbrales y Macro AP

Para cada una de las nueve clases repetimos matching y AP con los umbrales 0.50, 0.55, 0.60, 0.65, 0.70, 0.75 y 0.80. El AP de una clase es el promedio de sus siete resultados. El score Macro final es el promedio uniforme de las nueve clases, incluso cuando una clase no tiene GT y aporta 0.0.

In [ ]:
OFFICIAL_CLASS_IDS = tuple(range(1, 10))
OFFICIAL_RIOU_THRESHOLDS = (0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80)


def experimental_macro_ap_riou(
    predictions_by_class,
    ground_truths_by_class,
    class_ids=OFFICIAL_CLASS_IDS,
    thresholds=OFFICIAL_RIOU_THRESHOLDS,
):
    ap_by_class = {}
    ap_by_class_threshold = {}
    counts = {}

    for class_id in class_ids:
        predictions = predictions_by_class.get(class_id, [])
        ground_truths = ground_truths_by_class.get(class_id, [])
        threshold_aps = []

        for threshold in thresholds:
            matching = experimental_match_predictions(
                predictions, ground_truths, threshold
            )
            ap = experimental_ap_101(
                matching["tp_flags"], matching["fp_flags"], len(ground_truths)
            )[0]
            threshold_key = f"{threshold:.2f}"
            ap_by_class_threshold[(class_id, threshold_key)] = ap
            counts[(class_id, threshold_key)] = {
                "tp": matching["tp"],
                "fp": matching["fp"],
                "fn": matching["fn"],
            }
            threshold_aps.append(ap)

        ap_by_class[class_id] = float(np.mean(threshold_aps))

    macro_score = float(np.mean([ap_by_class[class_id] for class_id in class_ids]))
    details = {
        "ap_by_class": ap_by_class,
        "ap_by_class_threshold": ap_by_class_threshold,
        "counts": counts,
    }
    return macro_score, details


perfect_ground_truths = {}
perfect_predictions = {}
for class_id in OFFICIAL_CLASS_IDS:
    class_box = (100.0 + 5.0 * class_id, 100.0, 80.0, 40.0, 10.0)
    perfect_ground_truths[class_id] = [
        {
            "id": f"gt_class_{class_id}",
            "frame_id": "frame_perfect",
            "class_id": class_id,
            "obb": class_box,
        }
    ]
    perfect_predictions[class_id] = [
        {
            "id": f"pred_class_{class_id}",
            "frame_id": "frame_perfect",
            "class_id": class_id,
            "score": 1.0,
            "obb": class_box,
        }
    ]

perfect_macro, perfect_details = experimental_macro_ap_riou(
    perfect_predictions, perfect_ground_truths
)
assert math.isclose(perfect_macro, 1.0)
assert all(
    math.isclose(perfect_details["ap_by_class"][class_id], 1.0)
    for class_id in OFFICIAL_CLASS_IDS
)
print(f"Nueve clases perfectas: Macro AP={perfect_macro:.4f}")

## 18. Casos sintéticos completos

Verificamos las ambigüedades documentadas en el plan. Si solo dos clases son perfectas, Macro AP es 2/9 porque las otras siete aportan cero. Un duplicado posterior cuenta como FP pero puede conservar AP=1.0; para reducir AP colocamos un FP de mayor score antes del TP. También comprobamos predicciones vacías y desviación angular progresiva.

In [ ]:
two_class_predictions = {class_id: perfect_predictions[class_id] for class_id in (1, 2)}
two_class_ground_truths = {
    class_id: perfect_ground_truths[class_id] for class_id in (1, 2)
}
two_class_macro, _ = experimental_macro_ap_riou(
    two_class_predictions, two_class_ground_truths
)
empty_macro, _ = experimental_macro_ap_riou({}, perfect_ground_truths)

assert math.isclose(two_class_macro, 2.0 / 9.0)
assert math.isclose(empty_macro, 0.0)

angular_reference = (200.0, 150.0, 160.0, 40.0, 0.0)
angular_degrees = (10.0, 20.0, 30.0, 45.0)
angular_rious = [
    experimental_rotated_iou(
        angular_reference,
        (200.0, 150.0, 160.0, 40.0, angle),
    )[0]
    for angle in angular_degrees
]
assert all(
    angular_rious[index] > angular_rious[index + 1]
    for index in range(len(angular_rious) - 1)
)
assert angular_rious[-1] < 0.50

duplicate_gt = {1: ground_truths_example}
tp_then_duplicate_predictions = {
    1: [
        {**predictions_with_duplicate[0], "id": "tp_first", "score": 0.95},
        {**predictions_with_duplicate[0], "id": "duplicate_after", "score": 0.80},
    ]
}
fp_then_tp_predictions = {
    1: [
        {
            "id": "high_score_fp",
            "frame_id": "frame_001",
            "class_id": 1,
            "score": 0.99,
            "obb": separated_box,
        },
        {**predictions_with_duplicate[0], "id": "lower_score_tp", "score": 0.90},
    ]
}
duplicate_macro, duplicate_details = experimental_macro_ap_riou(
    tp_then_duplicate_predictions, duplicate_gt
)
early_fp_macro, early_fp_details = experimental_macro_ap_riou(
    fp_then_tp_predictions, duplicate_gt
)

assert math.isclose(duplicate_details["ap_by_class"][1], 1.0)
assert math.isclose(duplicate_macro, 1.0 / 9.0)
assert math.isclose(early_fp_details["ap_by_class"][1], 0.5)
assert math.isclose(early_fp_macro, 0.5 / 9.0)
for threshold in OFFICIAL_RIOU_THRESHOLDS:
    key = (1, f"{threshold:.2f}")
    assert duplicate_details["counts"][key] == {"tp": 1, "fp": 1, "fn": 0}

print(f"Solo clases 1 y 2 perfectas: Macro AP={two_class_macro:.4f} = 2/9")
print(f"Predicciones vacías: Macro AP={empty_macro:.4f}")
for angle, angle_riou in zip(angular_degrees, angular_rious):
    print(f"Desviación {angle:>4.0f}°: rIoU={angle_riou:.4f}")
print(
    "TP y luego duplicado: "
    f"AP clase 1={duplicate_details['ap_by_class'][1]:.2f}, "
    f"Macro={duplicate_macro:.4f}"
)
print(
    "FP antes del TP: "
    f"AP clase 1={early_fp_details['ap_by_class'][1]:.2f}, "
    f"Macro={early_fp_macro:.4f}"
)

## 19. Fundamentación científica y alcance de las referencias

**Ding et al. (2022), DOTA.** El benchmark DOTA demuestra la necesidad de representar objetos aéreos con orientaciones arbitrarias mediante cajas orientadas, especialmente cuando existen grandes cambios de escala y escenas densas. Esta evidencia respalda nuestra representación OBB y el cálculo geométrico de solapamiento entre polígonos orientados. DOI: `10.1109/TPAMI.2021.3117983`.

**Yang et al. (2021), KLD.** El análisis de regresión rotada muestra que centro, escala, relación de aspecto y ángulo están acoplados. En objetos alargados, un pequeño error angular puede perjudicar fuertemente la localización de alta precisión. Esto fundamenta el experimento de desviación angular progresiva y la evaluación con varios umbrales estrictos de rIoU. arXiv: `2106.01883`.

**Límite de esta fundamentación.** Estos trabajos respaldan el uso de OBB, la sensibilidad angular y la evaluación geométrica orientada, pero no se les atribuye la interpolación de 101 puntos. El matching greedy, los siete umbrales, el promedio de nueve clases y la interpolación COCO de 101 puntos corresponden al contrato específico del SMART Challenge documentado en `02_metric.md`.

## 20. Conclusiones finales del prototipo

El notebook ya recorre el pipeline matemático completo: OBB paramétrica, vértices, validación, normalización angular, rIoU, matching greedy, TP/FP/FN, Precision/Recall acumulados, AP COCO de 101 puntos y Macro AP sobre nueve clases y siete umbrales.

Resultados clave: nueve clases perfectas producen 1.0; solo dos clases perfectas producen 2/9; una clase sin GT produce AP 0.0; predicciones vacías producen 0.0; la desviación angular reduce rIoU; un duplicado posterior cuenta como FP aunque AP pueda conservarse en 1.0; un FP de mayor score antes del TP reduce AP a 0.5.

Las funciones experimentales ya fueron migradas de forma modular y tipada a `src/evaluation/metric.py`. Su contrato está cubierto por `test_metric.py`; el notebook permanece como evidencia explicativa del proceso experimental.